In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd

df_normalized = pd.read_csv('normalized_min-max.csv') # change csv file name
target_normalized = df_normalized['AgreeSubsequentBooster']


features_normalized = df_normalized.drop('AgreeSubsequentBooster', axis=1)

x_train_normalized, x_test_normalized, y_train_normalized, y_test_normalized = train_test_split(features_normalized,
                                                                                                target_normalized,
                                                                                                test_size=0.2,
                                                                                                random_state=0)

In [2]:
anova_columns = [
    'Age','EmploymentYears',

    'S.VaccineImportantHealth','S.VaccineEffectiveSevere','S.VaccineImportantPublic',
    'S.VaccineBeneficialMOH','S.VaccineSafe','S.MOHInfoTrusted',
    'S.VaccineProtectInfection','S.AgreeMOHRecommendation','S.WorriedSideEffects',
    'S.PreventNewVariant','P.ProtectFlu','P.ProtectSevereComp','P.EffectivenessVsRisk',
    'P.ClinicallyTested','P.LongTermSideEffects','P.TrustReportsMOH','P.ChipBelief',
    'P.MandatoryBelief','P.HalalDoubt','P.AlternativeMedicineBelief','P.HighRiskJob',
    'P.WorkDemand','A.BoosterProtectSevere','A.BoosterProtectFamily',
    'A.BoosterPreventSpread','A.ReturnDailyActivities','A.RecommendingBooster',
    'A.MaintainAntibody','A.NoSeriousSideEffects','B.RiskWithoutBooster',
    'B.SevereCompWithoutBooster','B.SpreadFamilyWithoutBooster',
    'B.TransmitPatientsWithoutBooster','B.SkepticalBooster','B.BoosterSafe',
    'B.BoosterMoreSideEffects','B.PreferNaturalImmunity','B.EliminateStigma',
    'B.PublicRiskWithoutBooster','C.BoosterNotNeeded','C.BoosterHarm',
    'C.EvidenceInsufficient','C.TrustSocialMedia','C.SideEffectsNoImpact'
]

chi2_columns = [
    'Gender','Education','Income','Pregnant','PatientContact','Occupation',
    'CovidPatientCare','PreExistingCondition','CovidInfected','Severity',
    'AdditionalVaccines','VaccinationStage','VaccinationSideEffects',
    'FamilySideEffects','SideEffectsAffectView',

    # one-hot race dummies
    'Race_Chinese','Race_Indian','Race_Malay','Race_Siam',

    # one-hot religion dummies
    'Religion_Buddha','Religion_Hindu','Religion_Islam',

    # one-hot location dummies
    'Location_Johor','Location_Kedah','Location_Kelantan','Location_Kuantan ',
    'Location_Melaka','Location_Negeri Sembilan','Location_Pahang',
    'Location_Perlis','Location_Pulau Pinang','Location_Sabah',
    'Location_Selangor','Location_Terengganu'
]


# Extract and save ANOVA columns from the training set
anova_train = x_train_normalized[anova_columns]

# Extract and save Chi-Squared columns from the training set
chi2_train = x_train_normalized[chi2_columns]

In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif, chi2

# ----------------------------
# Helper: drop constant columns
# ----------------------------
def drop_constant_columns(X):
    # Keep columns that have more than 1 unique value
    return X.loc[:, X.nunique(dropna=False) > 1]

# ----------------------------
# 1) ANOVA on numeric features
# ----------------------------
anova_train_clean = drop_constant_columns(anova_train)

anova_selector = SelectKBest(score_func=f_classif, k='all')
anova_selector.fit(anova_train_clean, y_train_normalized)

anova_scores = anova_selector.scores_
anova_pvals  = anova_selector.pvalues_

feature_scores_anova = pd.DataFrame({
    "Feature": anova_train_clean.columns,
    "Score": anova_scores,          # F-statistic
    "P-value": anova_pvals,
    "Method": "ANOVA"
})

# ----------------------------
# 2) Chi-square on categorical/non-negative features
# ----------------------------
chi2_train_clean = drop_constant_columns(chi2_train)

# IMPORTANT: chi2 requires non-negative values
if (chi2_train_clean.values < 0).any():
    raise ValueError("Chi-square requires non-negative features. Check your preprocessing for chi2 columns.")

chi_selector = SelectKBest(score_func=chi2, k='all')
chi_selector.fit(chi2_train_clean, y_train_normalized)

chi_scores = chi_selector.scores_     # chi-square statistic
chi_pvals  = chi_selector.pvalues_

feature_scores_chi = pd.DataFrame({
    "Feature": chi2_train_clean.columns,
    "Score": chi_scores,
    "P-value": chi_pvals,
    "Method": "Chi2"
})

# ----------------------------
# 3) Combine + sort baseline ranking
# ----------------------------
feature_scores = pd.concat([feature_scores_anova, feature_scores_chi], ignore_index=True)

# Drop any NaNs just in case
feature_scores = feature_scores.dropna(subset=["P-value"])

# Sort: smallest p-value = most important
feature_scores_sorted = feature_scores.sort_values(by="P-value", ascending=False).reset_index(drop=True)

print(feature_scores_sorted.head(30))

# Save baseline ranking
feature_scores_sorted.to_csv("p_values_for_base.csv", index=False)

                     Feature     Score   P-value Method
0      P.LongTermSideEffects  0.006263  0.937025  ANOVA
1                     Income  0.007363  0.931619   Chi2
2      Location_Pulau Pinang  0.013460  0.907640   Chi2
3                 Occupation  0.015176  0.901956   Chi2
4           VaccinationStage  0.032748  0.856396   Chi2
5             Religion_Islam  0.129844  0.718594   Chi2
6                 Race_Malay  0.129844  0.718594   Chi2
7             PatientContact  0.168930  0.681064   Chi2
8                   Severity  0.230138  0.631422   Chi2
9     VaccinationSideEffects  0.270150  0.603231   Chi2
10             P.HighRiskJob  0.291855  0.589800  ANOVA
11      PreExistingCondition  0.394503  0.529942   Chi2
12         Location_Kelantan  0.398744  0.527739   Chi2
13              P.ProtectFlu  0.615194  0.434021  ANOVA
14            Location_Johor  0.704404  0.401308   Chi2
15  B.BoosterMoreSideEffects  0.826919  0.364559  ANOVA
16  Location_Negeri Sembilan  0.848837  0.356881

In [4]:
feature_scores_sorted.to_csv('p_values_for_base.csv')

In [5]:
# decision tree shap values

import shap
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

# Train model
model_dt = DecisionTreeClassifier(max_depth=5, criterion='entropy', random_state=0)
model_dt.fit(x_train_normalized, y_train_normalized)

# SHAP (TreeExplainer)
explainer = shap.TreeExplainer(model_dt)

# Use the call-style API so we can handle Explanation objects cleanly
sv = explainer(x_test_normalized)

# sv can be:
# - shap.Explanation (common in newer SHAP)
# - list of arrays (older style)
# - ndarray
if hasattr(sv, "values"):   # Explanation
    values = sv.values
else:
    values = sv

# Now handle shapes:
# Case A: values is list [class0, class1]
if isinstance(values, list):
    shap_pos = values[1]   # class 1 matrix (n_samples, n_features)

# Case B: values is ndarray
else:
    # If 3D: (n_samples, n_features, n_classes) -> take class 1
    if values.ndim == 3:
        shap_pos = values[:, :, 1]
    # If 2D: already (n_samples, n_features)
    elif values.ndim == 2:
        shap_pos = values
    else:
        raise ValueError(f"Unexpected SHAP values shape: {values.shape}")

# Finally: mean absolute SHAP per feature -> 1D (n_features,)
shap_importance = np.abs(shap_pos).mean(axis=0)

# Sanity check (optional but helpful)
print("shap_pos shape:", np.shape(shap_pos))
print("shap_importance shape:", np.shape(shap_importance))
print("num features:", len(x_test_normalized.columns))

# Build ranking df
shap_rank_df = pd.DataFrame({
    "Feature": x_test_normalized.columns,
    "SHAP_Importance": shap_importance
}).sort_values("SHAP_Importance", ascending=False)

shap_rank_df.to_csv("Booster_dt_shap_output.csv", index=False)
print(shap_rank_df.head(20))

shap_pos shape: (40, 80)
shap_importance shape: (80,)
num features: 80
                       Feature  SHAP_Importance
41      A.BoosterPreventSpread         0.189198
2                    Education         0.126605
29       P.EffectivenessVsRisk         0.054848
38                P.WorkDemand         0.053484
24    S.AgreeMOHRecommendation         0.048655
16       SideEffectsAffectView         0.043169
12          AdditionalVaccines         0.031448
46        B.RiskWithoutBooster         0.027748
17    S.VaccineImportantHealth         0.027708
58      C.EvidenceInsufficient         0.019941
55  B.PublicRiskWithoutBooster         0.017438
52    B.BoosterMoreSideEffects         0.014952
34           P.MandatoryBelief         0.014348
33                P.ChipBelief         0.011480
13            VaccinationStage         0.009116
57               C.BoosterHarm         0.000000
56          C.BoosterNotNeeded         0.000000
0                          Age         0.000000
60       C.SideEf

In [6]:
import shap
import numpy as np
import pandas as pd
from sklearn.svm import SVC

# 1) Train SVC (make sure x_train/x_test are the SAME feature set)
model_svc = SVC(kernel="rbf", C=1, gamma="scale", probability=True, random_state=0)
model_svc.fit(x_train_normalized, y_train_normalized)

# 2) Sample background and explain set (keep as DataFrames)
background = shap.sample(x_train_normalized, 50, random_state=0)
X_explain = shap.sample(x_test_normalized, 200, random_state=0)

# 3) Predict function (keeps column alignment)
def model_predict_proba(X):
    X = pd.DataFrame(X, columns=x_train_normalized.columns)
    return model_svc.predict_proba(X)

# 4) KernelExplainer
explainer = shap.KernelExplainer(model_predict_proba, background)

sv = explainer.shap_values(X_explain, nsamples=200)

# 5) Robustly extract positive-class SHAP matrix
# sv can be list or ndarray
if isinstance(sv, list):
    shap_pos = sv[1]                         # (n_samples, n_features)
else:
    # If it’s 3D (n_samples, n_features, n_classes), take class 1
    shap_pos = sv[:, :, 1] if sv.ndim == 3 else sv

# 6) Importance = mean absolute SHAP per feature (must be length = n_features)
shap_importance = np.abs(shap_pos).mean(axis=0).ravel()

# 7) Sanity checks (prints help you confirm alignment)
print("Columns in X_explain:", X_explain.shape[1])
print("Length shap_importance:", len(shap_importance))

# 8) Build ranking using X_explain.columns (guaranteed same length)
shap_rank_df = pd.DataFrame({
    "Feature": X_explain.columns,
    "SHAP_Importance": shap_importance
}).sort_values("SHAP_Importance", ascending=False)

shap_rank_df.to_csv("Booster_svc_shap_output.csv", index=False)
print(shap_rank_df.head(20))

  0%|          | 0/40 [00:00<?, ?it/s]

Columns in X_explain: 80
Length shap_importance: 80
                             Feature  SHAP_Importance
53           B.PreferNaturalImmunity         0.031888
38                      P.WorkDemand         0.020370
2                          Education         0.020166
41            A.BoosterPreventSpread         0.013759
16             SideEffectsAffectView         0.013160
56                C.BoosterNotNeeded         0.011916
55        B.PublicRiskWithoutBooster         0.011398
35                      P.HalalDoubt         0.009462
43             A.RecommendingBooster         0.008876
18          S.VaccineEffectiveSevere         0.008817
17          S.VaccineImportantHealth         0.008710
40            A.BoosterProtectFamily         0.008384
58            C.EvidenceInsufficient         0.008027
46              B.RiskWithoutBooster         0.007463
47        B.SevereCompWithoutBooster         0.006392
26               S.PreventNewVariant         0.006092
33                      P.Chip